# Smart mode: fix or rewrite text with Qwen2.5-3B
Runtime > Change runtime type > **T4 GPU**. The first run downloads about 6 GB.

In [ ]:
!pip install -q transformers accelerate

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "Qwen/Qwen2.5-3B-Instruct"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(
    name, torch_dtype=torch.float16, device_map="auto"
)

PROMPTS = {
    "fix": "Fix the spelling and grammar of the text. Keep the same meaning and the same sentences as much as possible. Return only the corrected text.",
    "rewrite": "Rewrite the text in natural, vivid, fluent English with strong verbs and good flow. Fix all errors and keep the meaning. Return only the final text.",
}


def run(text, mode="fix"):
    messages = [
        {"role": "system", "content": PROMPTS[mode]},
        {"role": "user", "content": text},
    ]
    prompt = tok.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=False
    )
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=500, do_sample=False)
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tok.decode(new_tokens, skip_special_tokens=True)

In [ ]:
text = """PASTE YOUR MESSY TEXT HERE"""
print(run(text, "fix"))
print("-----")
print(run(text, "rewrite"))

Done? Runtime > **Disconnect and delete runtime**.